Edge-displacement outliers (descriptive, for limitations).

Pins with |outward_m - mean| above the 95th percentile from `edge_displacement.ipynb`.
Shows crop, Overture outlines (yellow), pin (red +), your click (cyan x). Tag the cause, labels are not changed.

In [1]:
%env DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib

import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from affine import Affine
from IPython.display import display
from rasterio.windows import Window
from shapely import from_wkb
from shapely.geometry import Point
from shapely.ops import nearest_points

from thesis.data import naip
from thesis.data.eligibility import context_for_scene
from thesis.run import REPO_ROOT, _geometry_to_crs, _valid_context_footprints, load_config

out_dir = REPO_ROOT / "data" / "dataset_v2"
imagery_config = load_config("imagery")
scene_cache = REPO_ROOT / imagery_config["scene_cache_dir"]
ZOOM_PX = 48

selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
boxes = gpd.read_parquet(out_dir / "inventory" / "boxes.parquet")
box_geometries = boxes.set_index("box_id").geometry.to_dict()
footprints = _valid_context_footprints(gpd.read_parquet(REPO_ROOT / load_config("overture")["out"]))

env: DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib


In [2]:
# same crop loading as gold_dataset_filtering / edge_displacement
item_records: dict[str, dict[str, dict]] = {}


def get_item(box_id: str, naip_item_id: str) -> dict:
    if box_id not in item_records:
        box_wgs84 = _geometry_to_crs(box_geometries[box_id], "EPSG:5070", "EPSG:4326")
        item_records[box_id] = {
            str(item["id"]): item for item in naip.search_items_for_geometry(box_wgs84, imagery_config)
        }
    return item_records[box_id][str(naip_item_id)]


def raw_crop_for(row) -> np.ndarray:
    item = get_item(str(row["box_id"]), row["naip_item_id"])
    scene_window = Window(*[int(v) for v in row["scene_window"]])
    cache_key = naip.window_cache_key(str(row["naip_item_id"]), scene_window)
    tile = naip.load_cached(scene_cache, cache_key)
    if tile is None:
        tile = naip.read_item_window(
            item, scene_window, float(imagery_config["gsd_m"]), float(imagery_config["gsd_tolerance"])
        )
        naip.store_atomic(scene_cache, cache_key, tile.array, tile.metadata)
    col_off, row_off, width, height = [int(v) for v in row["crop_window"]]
    return tile.array[:, row_off : row_off + height, col_off : col_off + width]


def scene_footprints(row) -> gpd.GeoDataFrame:
    scene_wgs84 = _geometry_to_crs(from_wkb(row["scene_geometry_wkb"]), "EPSG:5070", "EPSG:4326")
    return context_for_scene(footprints, scene_wgs84).to_crs(str(row["naip_source_crs"]))


def outward_m(row, union) -> float:
    pin = Point(float(row["tip_x"]), float(row["tip_y"]))
    reference = nearest_points(union.boundary, pin)[0]
    normal = np.array([reference.x - pin.x, reference.y - pin.y])
    normal /= np.linalg.norm(normal)
    label_error = float(np.array([row["click_x"] - reference.x, row["click_y"] - reference.y]) @ normal)
    return -label_error if row["exact_signed_distance_m"] > 0 else label_error

In [3]:
marks = pd.read_json(out_dir / "gold" / "edge_marks_phillip.jsonl", lines=True)
clicked = (
    marks.loc[~marks["skipped"], ["pin_id", "click_x", "click_y"]]
    .merge(pins, on="pin_id")
    .merge(
        selected[["source_id", "box_id", "naip_item_id", "naip_source_crs", "scene_window", "scene_geometry_wkb"]],
        on="source_id",
    )
)
clicked["footprints"] = [scene_footprints(r) for _, r in clicked.iterrows()]
clicked["outward_m"] = [outward_m(r, r["footprints"].union_all()) for _, r in clicked.iterrows()]
residual = (clicked["outward_m"] - clicked["outward_m"].mean()).abs()
outliers = clicked.loc[residual > residual.quantile(0.95)].sort_values("outward_m").reset_index(drop=True)
len(outliers), outliers[["pin_id", "bin_id", "exact_signed_distance_m", "outward_m"]]

(10,
                                               pin_id  bin_id  \
 0  d6ca3541920c5049bc44567442cd8a6dbd6aacb536f535...       1   
 1  9903f4cf376ca7c6f34bbca275dee50a512f9feca334f0...       1   
 2  5517f5e17ec05938591772fa6274645577aad6b7e078c5...      -1   
 3  90e9d57255ea46bfddf7dd855f9e1f7ddf66d678467755...      -1   
 4  335a430e394b9d7d09e3c2f0b7d144baefc79c30dce145...       1   
 5  838aff52a01bbfa549fd840bde147a6802eb02de142da2...       1   
 6  8818a7f10ff2d2fa8b8d719b2da1805b5b893f0dd07653...       1   
 7  27e7c53b85902dba7d0edcb07acb3b44fd800784ef74b1...       1   
 8  00b0c2aea7a5cf943e187cbf94839bbccb210aed1e3e72...      -1   
 9  c8ef8e6a405885f285ae6222c218f0031e6f973b446068...       1   
 
    exact_signed_distance_m  outward_m  
 0                 0.556113  -9.167512  
 1                 0.587202  -4.895985  
 2                -0.426677  -3.870943  
 3                -0.597872  -3.678529  
 4                 1.225654  -3.449162  
 5                 0.354639  -3.

In [4]:
log_path = out_dir / "gold" / "edge_outlier_causes.jsonl"
CAUSES = "registration / lean / footprint_shape / wrong_building / my_click / other"
j = 0

In [15]:
# run repeatedly
row = outliers.iloc[j]
crop = raw_crop_for(row)
transform = Affine(*row["crop_transform"])
to_px = ~transform  # world -> pixel corner coordinates; -0.5 for imshow pixel centres

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(np.transpose(crop, (1, 2, 0)))
for geometry in row["footprints"].geometry:
    for polygon in getattr(geometry, "geoms", [geometry]):
        cols, rows = zip(*[to_px * xy for xy in polygon.exterior.coords])
        ax.plot(np.array(cols) - 0.5, np.array(rows) - 0.5, color="yellow", linewidth=1)
click_col, click_row = to_px * (row["click_x"], row["click_y"])
ax.plot(row["tip_col"], row["tip_row"], "r+", markersize=14)
ax.plot(click_col - 0.5, click_row - 0.5, "cx", markersize=10)
ax.set_xlim(row["tip_col"] - ZOOM_PX, row["tip_col"] + ZOOM_PX)
ax.set_ylim(row["tip_row"] + ZOOM_PX, row["tip_row"] - ZOOM_PX)
ax.axis("off")
display(fig)
plt.close(fig)
print(
    f"[{j + 1}/{len(outliers)}] bin={row['bin_id']} exact={row['exact_signed_distance_m']:.2f} outward={row['outward_m']:.2f}"
)

cause = input(f"cause ({CAUSES}): ").strip()
with log_path.open("a") as f:
    f.write(json.dumps({"pin_id": row["pin_id"], "outward_m": float(row["outward_m"]), "cause": cause}) + "\n")
j += 1

IndexError: single positional indexer is out-of-bounds

In [ ]:
pd.read_json(log_path, lines=True)["cause"].value_counts()